<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/t03_docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Molecular docking with AutoDock Vina

The aim of this practical is to learn how to dock a small molecule onto a protein target, and above all **how to judge whether a docking result can be trusted**. The reference theory is that of Lecture 4:

*   https://github.com/carlocamilloni/Structural-Bioinformatics/blob/main/Notes/04_StructurePredictionDocking.pdf

We use a classic test case, *redocking*: we take a crystal structure of a protein with its ligand, remove the ligand, dock it back and check whether docking finds the experimental pose. Here the protein is **HIV-1 protease** and the ligand is **indinavir**, a drug used in the antiretroviral therapy of HIV (PDB entry [1HSG](https://www.rcsb.org/structure/1HSG)). Since we know the answer, we can measure how good the prediction is. In the report you will apply the same procedure to a system where you must work as if the answer were unknown.

## Introduction

**Molecular docking** explores the possible binding poses of a small molecule in the binding site of a protein of known (experimental or predicted) structure, and ranks them with a **scoring function**, an estimate of the binding free energy in kcal/mol.

<figure><center>
<img src='https://raw.githubusercontent.com/pb3lab/ibm3202/master/images/docking_01.png'/>
<figcaption>FIGURE 1. In molecular docking binding is evaluated in two steps: A) the transition of ligand and target from the unbound states to the conformations of the complex, and B) the energetics of binding in these conformations. <br> Huey R et al (2007) <i>J Comput Chem 28(6), 1145-1152.</i></figcaption></center></figure>

A docking program has two parts: a **search algorithm**, which generates poses (positions, orientations and conformations of the ligand) inside a box, and a **scoring function**, which ranks them. AutoDock Vina uses a Monte Carlo search with local optimisation, and an empirical scoring function made of steric, hydrophobic, hydrogen-bond and torsional terms. Two consequences to keep in mind:

*   the search is **stochastic**: different random seeds can give different results;
*   the score is an **approximation**: the best-scored pose is not necessarily the correct one, and the score is only a rough estimate of the affinity. (Vina's score, for example, does *not* use the partial charges of the atoms.)

<figure><center>
<img src='https://raw.githubusercontent.com/pb3lab/ibm3202/master/images/docking_02.png'/>
<figcaption>FIGURE 2. General steps of molecular docking: preparation of target and ligand, definition of the search space, docking, and analysis and validation of the poses.<br><i>Taken from Pars Silico (en.parssilico.com).</i></figcaption></center></figure>

**The tools**: [pdb2pqr](https://pdb2pqr.readthedocs.io) (adds hydrogens to the protein at a given pH, predicting the protonation states with PROPKA), [RDKit](https://www.rdkit.org) (chemistry: from a SMILES to a 3D molecule, RMSD), [Meeko](https://meeko.readthedocs.io) (prepares protein and ligand for AutoDock), [AutoDock Vina](https://autodock-vina.readthedocs.io) (docking: we download the official program, which works with any version of Python) and [py3Dmol](https://3dmol.org) (3D views). Everything except Vina installs with `pip`.

## Part 0. Setup

We work in a folder of the virtual machine (set `use_drive = True` to work in your Google Drive instead). The cell can be run more than once.

In [ ]:
import os, sys
if 'BASE_DIR' not in globals():          # remember where we started: running this cell again does not go one folder deeper
    BASE_DIR = os.getcwd()

use_drive = False        # True: keep your files on Google Drive (Colab will ask your permission)
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    workdir = '/content/drive/MyDrive/Structural_Bioinformatics/T03'
else:
    workdir = os.path.join(BASE_DIR, 'T03')
os.makedirs(workdir, exist_ok=True)
%cd $workdir

In [ ]:
# the Python packages we need, from PyPI (about one minute)
%pip install -q meeko rdkit pdb2pqr py3Dmol scipy gemmi

The next cell defines some functions for the analysis and the 3D views (double-click to read the code). The docking itself is done step by step in the cells below.

In [ ]:
#@title Analysis and visualisation helpers {display-mode: "form"}
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
from meeko import MoleculePreparation, PDBQTWriterLegacy, PDBQTMolecule, RDKitMolCreate
import py3Dmol
import os, re, sys, stat, platform, subprocess, urllib.request

def read_pdb_atoms(pdb_file, records=('ATOM',)):
    """Heavy atoms of a PDB file as a DataFrame (chain, residue, atom name, element, coordinates)."""
    rows = []
    for l in open(pdb_file):
        if l[:6].strip() in records:
            el = (l[76:78].strip() or l[12:16].strip()[0]).upper()
            if el == 'H': continue
            rows.append({'chain': l[21], 'resnum': int(l[22:26]), 'resname': l[17:20].strip(), 'atom': l[12:16].strip(),
                         'element': el, 'x': float(l[30:38]), 'y': float(l[38:46]), 'z': float(l[46:54])})
    return pd.DataFrame(rows)

def reference_ligand(pdb_file, resname, smiles):
    """The ligand of a crystal structure as an RDKit molecule, with bond orders and stereochemistry from a SMILES."""
    lines = [l for l in open(pdb_file) if l.startswith('HETATM') and l[17:20] == resname]
    mol = Chem.MolFromPDBBlock(''.join(lines) + 'END\n', removeHs=True)
    mol = AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(smiles), mol)
    Chem.AssignStereochemistryFrom3D(mol)
    return mol

def prepare_ligand(smiles, seed=42):
    """3D conformer of a SMILES (RDKit) and its PDBQT for Vina (Meeko)."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    if AllChem.EmbedMolecule(mol, randomSeed=seed) != 0:
        raise RuntimeError('RDKit could not build a 3D conformer for this SMILES')
    AllChem.MMFFOptimizeMolecule(mol)
    setup = MoleculePreparation().prepare(mol)[0]
    pdbqt, ok, err = PDBQTWriterLegacy.write_string(setup)
    if not ok:
        raise RuntimeError(err)
    return mol, pdbqt

def box_from_coordinates(xyz, padding=5.0):
    """Box enclosing a set of coordinates plus 'padding' Angstrom on each side."""
    xyz = np.asarray(xyz)
    return xyz.mean(0).round(2).tolist(), ((xyz.max(0) - xyz.min(0)) + 2*padding).round(1).tolist()

def box_from_residues(pdb_file, residues, padding=8.0):
    """Box around some residues, e.g. residues=[('A', 25), ('B', 25)]: useful when the ligand is not known."""
    at = read_pdb_atoms(pdb_file)
    sel = at[[(c, r) in set(residues) for c, r in zip(at['chain'], at['resnum'])]]
    if sel.empty:
        raise ValueError(f'none of the residues {residues} found in {pdb_file}')
    return box_from_coordinates(sel[['x', 'y', 'z']].values, padding)

VINA_VERSION = '1.2.7'

def get_vina(folder='bin'):
    """Download the AutoDock Vina program for this computer from its GitHub release (once) and return its path.
    We use the program and not the Python package because the program works with any version of Python."""
    system, machine = platform.system(), platform.machine().lower()
    if system == 'Linux':
        name = f'vina_{VINA_VERSION}_linux_' + ('aarch64' if machine in ('aarch64', 'arm64') else 'x86_64')
    elif system == 'Darwin':
        name = f'vina_{VINA_VERSION}_mac_' + ('aarch64' if machine in ('arm64', 'aarch64') else 'x86_64')
    elif system == 'Windows':
        name = f'vina_{VINA_VERSION}_win.exe'
    else:
        raise RuntimeError(f'no Vina program available for {system} {machine}')
    os.makedirs(folder, exist_ok=True)
    path = os.path.abspath(os.path.join(folder, name))
    if not os.path.exists(path):
        url = f'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v{VINA_VERSION}/{name}'
        urllib.request.urlretrieve(url, path)
        os.chmod(path, os.stat(path).st_mode | stat.S_IEXEC)
    return path

def box_arguments(center, size):
    return [f'--center_{a}={c}' for a, c in zip('xyz', center)] + [f'--size_{a}={s}' for a, s in zip('xyz', size)]

def read_scores(pdbqt_file):
    """The Vina scores (kcal/mol) of the poses written in an output file."""
    return np.array([float(l.split()[3]) for l in open(pdbqt_file) if l.startswith('REMARK VINA RESULT')])

def _vina(vina, args):
    r = subprocess.run([vina] + args, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'Vina failed:\n{r.stdout[-1500:]}\n{r.stderr[-1500:]}')
    return r.stdout

def run_docking(vina, receptor_pdbqt, ligand_pdbqt, center, size, seed=42, exhaustiveness=16, n_poses=9, out='poses.pdbqt',
                verbose=True):
    """Dock with AutoDock Vina (ligand given as a PDBQT string) and write the poses; returns their scores (kcal/mol)."""
    lig_file = out.replace('.pdbqt', '') + '_input_ligand.pdbqt'
    open(lig_file, 'w').write(ligand_pdbqt)
    stdout = _vina(vina, ['--receptor', receptor_pdbqt, '--ligand', lig_file, *box_arguments(center, size),
                          f'--seed={seed}', f'--exhaustiveness={exhaustiveness}', f'--num_modes={n_poses}', '--out', out])
    if verbose:
        print(stdout[stdout.find('mode |'):] if 'mode |' in stdout else stdout)
    return read_scores(out)

def read_poses(pdbqt_file):
    """The docked poses as a list of RDKit molecules (heavy atoms only), best score first."""
    mol = RDKitMolCreate.from_pdbqt_mol(PDBQTMolecule.from_file(pdbqt_file, skip_typing=True))[0]
    poses = []
    for conf in mol.GetConformers():
        p = Chem.Mol(mol); p.RemoveAllConformers(); p.AddConformer(Chem.Conformer(conf), assignId=True)
        poses.append(Chem.RemoveHs(p))
    return poses

def rmsd(mol_a, mol_b):
    """RMSD in place (no superposition) between two poses of the same molecule, symmetry-aware."""
    return rdMolAlign.CalcRMS(mol_a, mol_b)

def pose_table(poses, scores, reference=None):
    """Score of each pose, distance from the best-scored pose and (if given) from the reference."""
    rows = []
    for i, (p, s) in enumerate(zip(poses, scores)):
        row = {'pose': i + 1, 'score (kcal/mol)': round(float(s), 2), 'RMSD to pose 1 (A)': round(rmsd(p, poses[0]), 2)}
        if reference is not None:
            row['RMSD to crystal (A)'] = round(rmsd(p, reference), 2)
        rows.append(row)
    return pd.DataFrame(rows)

def plot_score_vs_rmsd(table, ax=None, label=None, threshold=2.0):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 3.8))
    ax.scatter(table['RMSD to crystal (A)'], table['score (kcal/mol)'], label=label)
    for _, r in table.iterrows():
        ax.annotate(int(r['pose']), (r['RMSD to crystal (A)'], r['score (kcal/mol)']), textcoords='offset points',
                    xytext=(4, 3), fontsize=8)
    ax.axvline(threshold, color='k', ls='--', lw=0.8)
    ax.axvspan(0, threshold, color='tab:green', alpha=0.1)
    ax.set_xlim(left=0)
    ax.set_xlabel('RMSD to the crystal pose (A)'); ax.set_ylabel('Vina score (kcal/mol)  (lower = better)')
    ax.text(threshold / 2, 0.02, 'correct', transform=ax.get_xaxis_transform(), ha='center', fontsize=8, color='darkgreen')
    return ax

def pose_rmsd_matrix(poses):
    n = len(poses)
    return np.array([[rmsd(poses[i], poses[j]) if i != j else 0.0 for j in range(n)] for i in range(n)])

def contacts(pose, protein_pdb, cutoff=4.0, polar_cutoff=3.5):
    """Protein residues within 'cutoff' of the ligand, and N/O pairs closer than 'polar_cutoff' (possible H-bonds)."""
    at = read_pdb_atoms(protein_pdb)
    P = at[['x', 'y', 'z']].values
    L = pose.GetConformer().GetPositions()
    lel = [a.GetSymbol() for a in pose.GetAtoms()]
    d = np.linalg.norm(P[:, None, :] - L[None, :, :], axis=2)
    rows = []
    for (ch, rn, rname), g in at.groupby(['chain', 'resnum', 'resname']):
        dd = d[g.index]
        if dd.min() > cutoff: continue
        polar = [f"{g.iloc[i]['atom']}-{lel[j]}{j+1} {dd[i, j]:.1f}"
                 for i, j in zip(*np.where(dd < polar_cutoff))
                 if g.iloc[i]['element'] in ('N', 'O') and lel[j] in ('N', 'O')]
        rows.append({'residue': f'{rname}{rn}', 'chain': ch, 'min distance (A)': round(float(dd.min()), 2),
                     'possible H-bonds (protein atom-ligand atom, A)': ', '.join(polar)})
    return pd.DataFrame(rows).sort_values('min distance (A)').reset_index(drop=True)

def view_box(protein_pdb, center, size, ligand=None, residues=None, width=800, height=450):
    """Protein (cartoon), search box (blue) and optionally a ligand and some residues (sticks)."""
    v = py3Dmol.view(width=width, height=height)
    v.addModel(open(protein_pdb).read(), 'pdb')
    v.setStyle({'cartoon': {'color': 'spectrum'}})
    if residues:
        for ch, rn in residues:
            v.addStyle({'chain': ch, 'resi': rn}, {'stick': {'colorscheme': 'greenCarbon'}})
    if ligand is not None:
        v.addModel(Chem.MolToMolBlock(ligand), 'mol'); v.setStyle({'model': 1}, {'stick': {'colorscheme': 'grayCarbon'}})
    v.addBox({'center': dict(zip('xyz', center)), 'dimensions': {'w': size[0], 'h': size[1], 'd': size[2]},
              'color': 'blue', 'opacity': 0.4})
    v.setBackgroundColor('white'); v.zoomTo()
    return v

def view_poses(protein_pdb, poses, reference=None, labels=None, width=800, height=450, pocket_residues=None):
    """Protein, crystal ligand (grey) and docked poses (green, magenta, orange, ...)."""
    colors = ['greenCarbon', 'magentaCarbon', 'orangeCarbon', 'cyanCarbon', 'yellowCarbon']
    v = py3Dmol.view(width=width, height=height)
    v.addModel(open(protein_pdb).read(), 'pdb')
    v.setStyle({'model': 0}, {'cartoon': {'color': 'lightgrey', 'opacity': 0.8}})
    if pocket_residues is not None:
        for _, r in pocket_residues.iterrows():
            v.addStyle({'model': 0, 'chain': r['chain'], 'resi': int(''.join(c for c in r['residue'] if c.isdigit()))},
                       {'stick': {'radius': 0.12}})
    k = 1
    if reference is not None:
        v.addModel(Chem.MolToMolBlock(reference), 'mol'); v.setStyle({'model': k}, {'stick': {'colorscheme': 'grayCarbon'}}); k += 1
    for i, p in enumerate(poses):
        v.addModel(Chem.MolToMolBlock(p), 'mol'); v.setStyle({'model': k}, {'stick': {'colorscheme': colors[i % len(colors)]}}); k += 1
    v.setBackgroundColor('white'); v.zoomTo({'model': 1})
    return v

def ligand_conformers(smiles, n=10, seed=42):
    """Several 3D conformers of a SMILES (RDKit): docking does not change ring conformations, so starting from
    different conformers is the way to explore them. Returns the molecule (with hydrogens) holding all conformers."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    cids = AllChem.EmbedMultipleConfs(mol, numConfs=n, randomSeed=seed)
    if not cids:
        raise RuntimeError('RDKit could not build 3D conformers for this SMILES')
    AllChem.MMFFOptimizeMoleculeConfs(mol)
    return mol

def pdbqt_of_conformer(mol, conf_id):
    """PDBQT (Meeko) of one conformer of a molecule."""
    m = Chem.Mol(mol); m.RemoveAllConformers(); m.AddConformer(Chem.Conformer(mol.GetConformer(conf_id)), assignId=True)
    pdbqt, ok, err = PDBQTWriterLegacy.write_string(MoleculePreparation().prepare(m)[0])
    if not ok:
        raise RuntimeError(err)
    return pdbqt

def ring_conformations(mol, reference):
    """For every non-aromatic ring: RMSD (ring atoms and their neighbours, after superposition) between each conformer
    of mol and the reference. Values around 0.1 A mean the same ring conformation as the reference."""
    ref = Chem.RemoveHs(reference)
    rings = [r for r in ref.GetRingInfo().AtomRings() if not all(ref.GetAtomWithIdx(a).GetIsAromatic() for a in r)]
    heavy = Chem.RemoveHs(mol)
    match = heavy.GetSubstructMatch(ref, useChirality=False)
    rows = []
    for cid in [c.GetId() for c in heavy.GetConformers()]:
        row = {'conformer': cid}
        for r in rings:
            env = set(r)
            for a in r:
                env |= {n.GetIdx() for n in ref.GetAtomWithIdx(a).GetNeighbors()}
            het = sorted(ref.GetAtomWithIdx(a).GetSymbol() for a in r if ref.GetAtomWithIdx(a).GetSymbol() != 'C')
            name = f"{len(r)}-membered ring" + (f" ({'+'.join(het)})" if het else '')
            while name in row: name += "'"
            row[name] = round(rdMolAlign.AlignMol(Chem.Mol(heavy), ref, prbCid=cid, atomMap=[(match[i], i) for i in sorted(env)]), 2)
        rows.append(row)
    return pd.DataFrame(rows).set_index('conformer')

def score_reference(vina, receptor_pdbqt, reference, center, size):
    """Vina score of the experimental pose, as it is and after a local optimisation. If it is better than the best
    docked score, docking failed in the SEARCH; if it is worse, the SCORING function prefers a wrong pose."""
    pdbqt, ok, err = PDBQTWriterLegacy.write_string(MoleculePreparation().prepare(Chem.AddHs(reference, addCoords=True))[0])
    open('reference_ligand.pdbqt', 'w').write(pdbqt)
    common = ['--receptor', receptor_pdbqt, '--ligand', 'reference_ligand.pdbqt', *box_arguments(center, size)]
    energy = lambda out: float(re.search(r'Estimated Free Energy of Binding\s*:\s*(-?[0-9.]+)', out).group(1))
    as_is = energy(_vina(vina, common + ['--score_only']))
    optimised = energy(_vina(vina, common + ['--local_only', '--out', 'reference_ligand_optimised.pdbqt']))
    return round(as_is, 2), round(optimised, 2)

In [ ]:
VINA = get_vina()          # the AutoDock Vina program, downloaded once from its GitHub release
import rdkit, meeko
print('Python', sys.version.split()[0], '| RDKit', rdkit.__version__, '| Meeko', meeko.__version__)
!{VINA} --version

## Part 1. The receptor

We download the structure **1HSG** from the Protein Data Bank. Note: the title of the 1994 paper speaks of "HIV II protease", but the protein in this entry is the protease of **HIV-1**.

In [ ]:
import urllib.request
urllib.request.urlretrieve('https://files.rcsb.org/download/1HSG.pdb', '1HSG.pdb')

at = read_pdb_atoms('1HSG.pdb', records=('ATOM', 'HETATM'))
print('chains:', sorted(at['chain'].unique()))
print('residues that are not amino acids (HETATM):')
print(at[~at['resname'].isin(['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'])]
      .groupby('resname').size().rename('atoms'))

The file contains the two chains of the protease (it is a homodimer, the active site is between the two chains), the ligand indinavir (residue name **MK1**) and many **crystallographic water** molecules (HOH). For docking we keep only the protein: the `ATOM` lines (and the `TER` lines that separate the chains).

Most waters are not important for docking, but some are. In HIV-1 protease a single water molecule, the **flap water**, bridges the inhibitor and the two *flaps* (the flexible β-hairpins that close over the active site, residue Ile50 of each chain). With `keep_flap_water = True` you can keep it and test whether it matters (Part 6).

In [ ]:
keep_flap_water = False     # True: keep the water HOH 308 of chain B, bridging the ligand and the two Ile50

with open('1HSG_protein.pdb', 'w') as f:
    for line in open('1HSG.pdb'):
        if line.startswith('ATOM') or line.startswith('TER'):
            f.write(line)
        elif keep_flap_water and line.startswith('HETATM') and line[17:20] == 'HOH' and line[21] == 'B' and line[22:26].strip() == '308':
            f.write(line)
    f.write('END\n')
print('protein atoms:', len(read_pdb_atoms('1HSG_protein.pdb')))

**Hydrogens and protonation.** X-ray structures have no hydrogens, but docking needs the polar ones: they define which atoms can donate hydrogen bonds. Which hydrogens to add depends on the **protonation state** of each residue at the pH of interest. `pdb2pqr` predicts it with PROPKA (here at pH 7.4) and writes a PQR file (a PDB file with charges and radii).

In [ ]:
!pdb2pqr --ff=AMBER --ffout=AMBER --keep-chain --titration-state-method=propka --with-ph=7.4 1HSG_protein.pdb 1HSG_protein.pqr > pdb2pqr.log 2>&1
!tail -n 3 pdb2pqr.log

# AMBER names: ASP = deprotonated (charged) aspartate, ASH = protonated (neutral) aspartate
pqr = [l for l in open('1HSG_protein.pqr') if l.startswith('ATOM')]
print('the two catalytic aspartates:', sorted({(l[17:20], l[21], l[22:26].strip()) for l in pqr if l[22:26].strip() == '25'}))

# pdb2pqr calls the water WAT, Meeko knows it as HOH
txt = open('1HSG_protein.pqr').read()
txt = txt.replace(' OW  WAT ', ' O   HOH ').replace(' HW  WAT ', ' H1  HOH ', 1).replace(' HW  WAT ', ' H2  HOH ', 1)
open('1HSG_protein.pqr', 'w').write(txt)

Look at the two **catalytic aspartates**, Asp25 of chain A and Asp25 of chain B: they are chemically identical, but PROPKA protonates only *one* of them. The two carboxylates face each other in the active site and cannot both be charged; in complexes with inhibitors one of them is indeed protonated. A small detail with consequences: it decides which hydrogen bonds the ligand can make.

Now **Meeko** converts the receptor into the **PDBQT** format: a PDB with the partial charge (*q*) and the AutoDock atom type (*t*) of each atom. Non-polar hydrogens are merged into their carbon atoms.

In [ ]:
!mk_prepare_receptor.py --read_pqr 1HSG_protein.pqr -o 1HSG_receptor -p > meeko_receptor.log 2>&1
!tail -n 2 meeko_receptor.log
!head -n 4 1HSG_receptor.pdbqt

> **Questions.** Why do we need the hydrogens? Why only the polar ones? Would a different pH change anything in the active site?

## Part 2. The ligand

We start from a **SMILES** string, a line notation of the chemical structure, as you would do for a molecule that has never been crystallised. We download it from **PubChem**. Careful: since July 2025 PubChem gives two different SMILES, the *connectivity* SMILES (which atoms are bonded, **without stereochemistry**) and the *SMILES* that includes stereochemistry (the `@`/`@@` symbols). For a drug this difference is essential.

A reminder of the SMILES notation: `-` single, `=` double, `#` triple bond; lowercase letters (`c1ccccc1`) aromatic atoms; numbers open and close rings; parentheses are branches; `@` and `@@` the two configurations of a stereocentre; `/` and `\` the cis/trans configuration of a double bond.

In [ ]:
base = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/indinavir/property'
try:
    smiles = urllib.request.urlopen(f'{base}/SMILES/TXT', timeout=20).read().decode().split()[0]
    flat_smiles = urllib.request.urlopen(f'{base}/ConnectivitySMILES/TXT', timeout=20).read().decode().split()[0]
except Exception as e:
    print('PubChem not reachable, using the stored SMILES:', e)
    smiles = 'CC(C)(C)NC(=O)[C@@H]1CN(Cc2cccnc2)CCN1C[C@@H](O)C[C@@H](Cc1ccccc1)C(=O)N[C@H]1c2ccccc2C[C@H]1O'
    flat_smiles = 'CC(C)(C)NC(=O)C1CN(CCN1CC(CC(CC2=CC=CC=C2)C(=O)NC3C(CC4=CC=CC=C34)O)O)CC5=CN=CC=C5'
print('SMILES             :', smiles)
print('connectivity SMILES:', flat_smiles)

for name, s in [('SMILES', smiles), ('connectivity SMILES', flat_smiles)]:
    centres = Chem.FindMolChiralCenters(Chem.MolFromSmiles(s), includeUnassigned=True, useLegacyImplementation=False)
    print(f'{name}: stereocentres', centres)

Indinavir has **five stereocentres**. The connectivity SMILES leaves all of them unassigned (`?`): a 3D structure built from it could be any of 2⁵ = 32 stereoisomers, only one of which is the drug. The full SMILES fixes all of them (R or S).

Now we build a 3D structure from the SMILES with RDKit (adding the hydrogens and optimising the geometry with a force field) and we look at it. Note that this conformation has *nothing to do* with the crystal pose: docking will have to find the right one.

In [ ]:
ligand_mol, ligand_pdbqt = prepare_ligand(smiles, seed=42)
open('indinavir.pdbqt', 'w').write(ligand_pdbqt)
print('atoms (with hydrogens):', ligand_mol.GetNumAtoms())
print([l for l in ligand_pdbqt.splitlines() if l.startswith('TORSDOF')][0], ' <- number of rotatable bonds explored by docking')

view = py3Dmol.view(width=400, height=300)
view.addModel(Chem.MolToMolBlock(ligand_mol), 'mol'); view.setStyle({'stick': {}}); view.zoomTo(); view.show()

`prepare_ligand` used **Meeko** to write the PDBQT of the ligand: atom types, charges and, importantly, the **rotatable bonds** (the torsion tree): the docking search moves the ligand as a rigid body *and* rotates these bonds. Fourteen rotatable bonds is a lot: the more flexible the ligand, the harder the search.

Note what is **not** in the torsion tree: the bonds inside **rings**. Docking rotates the rotatable bonds, but it never changes the conformation of a ring (the chair of the piperazine, the pucker of the five-membered ring of the indane): these stay exactly as RDKit built them. Keep this in mind for Part 6.

> **Questions.** Which bonds are rotatable and which are not (amides, rings)? RDKit built the neutral molecule: is that the right protonation state at pH 7.4? Which atoms would change?

## Part 3. The search space

Vina searches for poses inside a **box**. The box must contain the whole binding site, with some room around it: too small and the ligand does not fit, too large and the search becomes slow and less reliable.

In this tutorial we know where the ligand is, so we build the box around the crystal ligand plus 5 Å on each side. **In your exercise you will not know it**: you can centre the box on residues of the binding site (`box_from_residues`, see below), or on a pocket predicted by a program like P2Rank. Here we also load the crystal ligand, with its bond orders and stereochemistry taken from the SMILES, as the reference to judge the results.

In [ ]:
reference = reference_ligand('1HSG.pdb', 'MK1', smiles)
center, size = box_from_coordinates(reference.GetConformer().GetPositions(), padding=5.0)
print('box centre:', center, ' box size:', size)

# the alternative, when the ligand is not known: a box around the catalytic aspartates and the flap tips
print('box from residues:', box_from_residues('1HSG_protein.pdb', [('A', 25), ('B', 25), ('A', 50), ('B', 50)], padding=8.0))

view_box('1HSG_protein.pdb', center, size, ligand=reference, residues=[('A', 25), ('B', 25), ('A', 50), ('B', 50)]).show()

## Part 4. Docking

We run the Vina program with its options: the receptor and ligand files, the box (centre and size), the random seed, the exhaustiveness, the number of poses and the output file. The cell prints the command, so you can see exactly what is run.

`exhaustiveness` controls how much the search tries (the default is 8; higher is slower but more thorough). With the settings below the docking takes a few minutes on Colab.

In [ ]:
seed = 42
exhaustiveness = 16

cmd = (f'{VINA} --receptor 1HSG_receptor.pdbqt --ligand indinavir.pdbqt '
       f'--center_x {center[0]} --center_y {center[1]} --center_z {center[2]} '
       f'--size_x {size[0]} --size_y {size[1]} --size_z {size[2]} '
       f'--seed {seed} --exhaustiveness {exhaustiveness} --num_modes 9 --out indinavir_poses.pdbqt')
print(cmd)
!{cmd}
scores = read_scores('indinavir_poses.pdbqt')

Vina prints a table with the poses (*modes*) ranked by **score** (affinity, kcal/mol: more negative is better) and two RMSD values from the best pose:

*   **rmsd u.b.** (*upper bound*) matches every atom with *itself* in the other pose;
*   **rmsd l.b.** (*lower bound*) matches every atom with the *closest* atom of the same type in the other pose.

The real distance between two poses lies between the two. When **l.b. is small and u.b. is large**, the two poses occupy the same region of space but the molecule is **turned around**: keep it in mind.

## Part 5. How good are the poses?

Here we can do what is impossible in a real application: compare every pose with the experimental one. The **RMSD to the crystal pose** is computed in place (no superposition, since protein and ligand are in the same reference frame), taking into account the symmetry of the molecule. By convention a pose within **2 Å** of the experimental one is considered **correct**.

In [ ]:
poses = read_poses('indinavir_poses.pdbqt')
table = pose_table(poses, scores, reference)
plot_score_vs_rmsd(table)
plt.show()
table

> **Questions.** Is the best-scored pose (pose 1) correct? If not, is there a correct pose further down the list, and how different is its score? Look at the column *RMSD to pose 1*: which poses are essentially the same, and which are different?

Let's look at them: in grey the crystal ligand, in green the best-scored pose, in magenta the pose closest to the crystal. The residues around the best pose are shown as sticks.

In [ ]:
best = 0
closest = int(table['RMSD to crystal (A)'].idxmin())
print(f'best-scored pose: {best + 1}, closest to the crystal: {closest + 1}')
view_poses('1HSG_protein.pdb', [poses[best], poses[closest]], reference,
           pocket_residues=contacts(poses[best], '1HSG_protein.pdb')).show()

HIV-1 protease is a **symmetric** dimer, and indinavir is roughly symmetric too: a pose with the molecule **turned around** fills the same cavity, makes similar contacts and can get a similar score (in the Vina table such poses have a small *l.b.* but a large *u.b.* RMSD). If your best-scored pose is not the correct one, this is the most important lesson of docking: **a good score is not proof of a correct pose**.

**Was it the search or the score?** A docking can fail in two ways: the *search* did not find the correct pose, or the *scoring function* prefers a wrong one. Here we can tell them apart, because we know the correct pose: we ask Vina to score the crystal pose (and to optimise it locally, to remove small clashes).

In [ ]:
xtal_as_is, xtal_optimised = score_reference(VINA, '1HSG_receptor.pdbqt', reference, center, size)
print(f'crystal pose: {xtal_as_is} kcal/mol as it is, {xtal_optimised} kcal/mol after local optimisation')
print(f'best docked pose: {scores[0]:.2f} kcal/mol (RMSD to crystal {table.loc[0, "RMSD to crystal (A)"]} A)')
if table.loc[0, 'RMSD to crystal (A)'] <= 2:
    print('-> the best-scored pose is correct')
elif xtal_optimised < scores[0]:
    print('-> the crystal pose scores BETTER than everything docking found: the SEARCH failed')
else:
    print('-> the crystal pose scores WORSE than the docked pose: the SCORING function prefers a wrong pose')

Finally, the **interactions**: which residues are in contact with a pose, and which pairs of polar atoms (N, O) are close enough for a hydrogen bond. Compare the correct pose with the best-scored one. (For a complete analysis of the interactions you can upload a complex to the [PLIP server](https://plip-tool.biotec.tu-dresden.de).)

In [ ]:
print('pose closest to the crystal:')
display(contacts(poses[closest], '1HSG_protein.pdb'))
print('best-scored pose:')
display(contacts(poses[best], '1HSG_protein.pdb'))

> **Questions.** Which residues interact with the ligand in the correct pose? Are the catalytic aspartates (Asp25 A and B) and the flaps (Gly48, Ile50) involved? Which interactions are lost, or gained, in the best-scored pose?

## Part 6. Experiments

A docking result should be **tested**, not just accepted. Each experiment repeats the docking with one change (a few minutes each). Set the flag to `True` to run it, and compare with the table of Part 5.

**6a. Convergence.** The search is stochastic: does another random seed give the same answer? Are the top poses found every time?

In [ ]:
run_6a = False
if run_6a:
    for s in [1, 7]:
        sc = run_docking(VINA, '1HSG_receptor.pdbqt', ligand_pdbqt, center, size, seed=s, exhaustiveness=exhaustiveness, out=f'poses_seed{s}.pdbqt')
        print(f'seed {s}'); display(pose_table(read_poses(f'poses_seed{s}.pdbqt'), sc, reference))

**6b. Ring conformations.** As we saw in Part 2, docking keeps the rings exactly as they were built. If RDKit built a ring in a different conformation from the one in the complex, the correct pose *cannot be reached*, however long the search. Let's build several conformers and compare their rings with the crystal (we can, because here we know the answer): values around 0.1 Å mean the same ring conformation as in the crystal, values of 0.4-0.7 Å a different one (a different pucker, or a substituent axial instead of equatorial).

In [ ]:
conformers = ligand_conformers(smiles, n=10, seed=42)
rings = ring_conformations(conformers, reference)
print('rings of the conformer docked in Part 4:')
display(ring_conformations(ligand_mol, reference))
print('rings of 10 new conformers:')
display(rings)
like_crystal = rings[(rings < 0.25).all(axis=1)].index.tolist()
print('conformers with all the rings as in the crystal:', like_crystal)

Now we dock one conformer with crystal-like rings and compare it with Part 4. In a real application we would not know which rings are right: the standard practice is to **dock several starting conformers** (with different ring conformations) and compare their best scores.

In [ ]:
run_6b = False
if run_6b and like_crystal:
    c = like_crystal[0]
    sc = run_docking(VINA, '1HSG_receptor.pdbqt', pdbqt_of_conformer(conformers, c), center, size, seed=seed, exhaustiveness=exhaustiveness, out='poses_rings.pdbqt')
    print(f'conformer {c}, rings as in the crystal:')
    display(pose_table(read_poses('poses_rings.pdbqt'), sc, reference))
    print(f'best score in Part 4: {scores[0]:.2f} kcal/mol, with these rings: {sc[0]:.2f} kcal/mol')
elif run_6b:
    print('no conformer with crystal-like rings: build more conformers (n=20, 50...)')

> **Questions.** Did the rings of the Part 4 conformer match the crystal? With crystal-like rings, is the best-scored pose correct? Compare the best scores of the two dockings: would the score alone have told you which starting conformer was right? Is this consistent with the answer to *search or score?* in Part 5?

**6c. Stereochemistry.** What happens if we dock the molecule built from the connectivity SMILES, without stereochemistry? RDKit picks the configuration of the stereocentres for us. Can a wrong stereoisomer still get a good score?

In [ ]:
run_6c = False
if run_6c:
    flat_mol, flat_pdbqt = prepare_ligand(flat_smiles, seed=42)
    Chem.AssignStereochemistryFrom3D(flat_mol)        # which configuration did RDKit build?
    built = Chem.RemoveHs(flat_mol)
    match = built.GetSubstructMatch(reference, useChirality=False)      # same atom numbering as the drug
    built_cip = dict(Chem.FindMolChiralCenters(built))
    print('stereocentre   indinavir   built by RDKit')
    for i, label in Chem.FindMolChiralCenters(reference):
        print(f'atom {i+1:>3}          {label}            {built_cip.get(match[i], "?")}')
    sc = run_docking(VINA, '1HSG_receptor.pdbqt', flat_pdbqt, center, size, seed=seed, exhaustiveness=exhaustiveness, out='poses_flat.pdbqt')
    flat_poses = read_poses('poses_flat.pdbqt')
    for p in flat_poses: Chem.AssignStereochemistryFrom3D(p)
    # the RMSD to the crystal is defined only if the stereoisomer is the same molecule
    same = Chem.MolToSmiles(flat_poses[0]) == Chem.MolToSmiles(reference)
    print('same stereoisomer as the drug:', same)
    display(pose_table(flat_poses, sc, reference if same else None))

**6d. The flap water.** Go back to Part 1, set `keep_flap_water = True`, and run again Parts 1 to 5 (the receptor file now contains the water). How do the scores change? Does the water change the ranking of the poses? Score the crystal pose again (*search or score?*): what does the water add?

**6e. The box.** Repeat the docking with a box built from residues (`box_from_residues`) instead of the crystal ligand, as you will have to do in the exercise, or with a larger padding. Does the answer change? How long does it take?

**6f. Exhaustiveness.** Try `exhaustiveness = 4` and `32`. Is the best score lower? Are the poses more reproducible?

## Part 7. Towards your exercise

In the report (`report_3_docking.ipynb`) you will dock a drug onto a protein structure **that does not contain it**, so you cannot measure the RMSD to a crystal pose. You will need to:

1.  **find the binding site**: from the literature, from conserved residues, or from a pocket prediction such as [P2Rank](https://prankweb.cz/) (upload the PDB file, look at the ranked pockets and their residues);
2.  **build the box** around it (`box_from_residues` with the residues of the pocket);
3.  **prepare** receptor and ligand as here (remove what does not belong, think about the protonation states, and use a SMILES *with* stereochemistry);
4.  **dock with several seeds and several starting conformers** (6a, 6b) and check the convergence, since there will be no crystal to tell you whether the top pose is right;
5.  **judge the poses** with the arguments you can use without a reference: reproducibility, the score gap between different binding modes, the interactions, and consistency with what is known (mutations, homologous complexes).

Copy the cells of this notebook that you need into your report.